# MAUDE

Sequential exploration of FDA MAUDE adverse-event reports.

## 1. Fetch

Source: [openFDA device/event bulk downloads](https://open.fda.gov/data/downloads/). This is the full MAUDE database as JSON, re-exported weekly and split into quarterly zip files (~19 GB zipped). It uses the same schema as the [`/device/event` API](https://open.fda.gov/apis/device/event/).

Download it with `uv run python scripts/download_maude.py`. The script resumes where it left off and puts each export in its own directory under `data/maude/raw/`. The cells below use the newest export.

In [ ]:
import json
import re
import zipfile
from collections import Counter
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pnpm-workspace.yaml').exists())
MAUDE_DIR = ROOT / 'data' / 'maude'
CLEAN_DIR = MAUDE_DIR / 'cleaned'

In [ ]:
RAW_DIR = max((MAUDE_DIR / 'raw').iterdir())
event = json.loads((RAW_DIR / 'manifest.json').read_text())

parts = pd.DataFrame(event['partitions']).astype({'size_mb': float, 'records': int})
parts['period'] = parts['file'].str.split('/').str[-2]  # e.g. 2004q3, all_other
parts['path'] = parts['file'].map(lambda u: RAW_DIR / u.split('/device/event/')[1])
parts = parts.sort_values('file', ignore_index=True)

missing = parts[~parts.path.map(Path.exists)]
if not missing.empty:
    raise FileNotFoundError(f'{len(missing)} files missing; re-run scripts/download_maude.py')

print(
    f'export {event["export_date"]}: {event["total_records"]:,} reports, '
    f'{len(parts)} files, {parts.size_mb.sum() / 1024:.1f} GB zipped'
)
parts.groupby(parts.period.str[:4]).agg(
    files=('file', 'size'), records=('records', 'sum'), size_mb=('size_mb', 'sum')
)

### Sanity check

Load one partition to look at the raw shape. Each zip holds a single JSON file, `{"meta": ..., "results": [report, ...]}`.

In [ ]:
def load_partition(path: Path) -> list[dict]:
    with zipfile.ZipFile(path) as zf:
        return json.loads(zf.read(zf.namelist()[0]))['results']


latest = parts[parts.period.str.match(r'\d{4}q\d')].iloc[-1]
raw = load_partition(latest.path)
print(latest.period, f'{len(raw):,} reports')
print('nested fields:', sorted(k for k, v in raw[0].items() if isinstance(v, list | dict)))

### Report context

The useful context is nested, so flatten it to one row per report:

- **item**: `device[]`: generic/brand name, manufacturer, model, product code, who operated it
- **failure**: `product_problems[]`, `patient[].patient_problems[]`, outcomes, `event_type`
- **narrative**: `mdr_text[]`: the event description and the manufacturer's follow-up
- **reporter**: `source_type[]`, `report_source_code`, `reporter_occupation_code`, patient age and sex

In [ ]:
def nonempty(xs: list[str]) -> list[str]:
    return [x for x in xs if x]


def flatten(r: dict) -> dict:
    # Nearly every report has exactly one device and one patient.
    dev = (r.get('device') or [{}])[0]
    pat = (r.get('patient') or [{}])[0]
    text: dict[str, list[str]] = {}
    for t in r.get('mdr_text', []):
        text.setdefault(t['text_type_code'], []).append(t['text'])
    return {
        'report_number': r.get('report_number'),
        'date_received': r.get('date_received'),
        'event_type': r.get('event_type'),
        'product_code': dev.get('device_report_product_code'),
        'generic_name': dev.get('generic_name'),
        'brand_name': dev.get('brand_name'),
        'manufacturer': dev.get('manufacturer_d_name'),
        'model_number': dev.get('model_number'),
        'device_operator': dev.get('device_operator'),
        'product_problems': nonempty(r.get('product_problems', [])),
        'patient_problems': nonempty(pat.get('patient_problems', [])),
        'outcomes': nonempty(pat.get('sequence_number_outcome', [])),
        'event_description': '\n\n'.join(text.get('Description of Event or Problem', [])),
        'mfr_narrative': '\n\n'.join(text.get('Additional Manufacturer Narrative', [])),
        'source_type': nonempty(r.get('source_type', [])),
        'report_source': r.get('report_source_code'),
        'reporter_occupation': r.get('reporter_occupation_code'),
        'patient_age': pat.get('patient_age'),
        'patient_sex': pat.get('patient_sex'),
    }


reports = pd.DataFrame(map(flatten, raw))
reports.map(bool).mean().mul(100).round().astype(int).rename('% non-empty')

### Insulin pumps

The same view limited to insulin devices, matched loosely on generic name for now.

In [ ]:
PUMP = re.compile(r'(?=.*insulin)(?=.*pump)', re.IGNORECASE)
COLUMNS = [
    'generic_name',
    'brand_name',
    'event_type',
    'event_description',
    'product_problems',
    'patient_problems',
    'outcomes',
    'device_operator',
    'source_type',
    'reporter_occupation',
]

pumps = reports[reports.generic_name.str.contains(PUMP, na=False)]
print(f'{len(pumps):,} of {len(reports):,} reports')

for col in COLUMNS:
    display(pumps[col].explode().value_counts().head(10))

In [ ]:
for r in pumps[pumps.event_description != ''].sample(5, random_state=0).itertuples():
    print(f'{r.report_number} | {r.brand_name} | {r.event_type} | {r.product_problems}')
    print(f'{r.event_description}\n')

## 2. Extract

Write every insulin-pump report (any device whose generic name matches `PUMP`) with all of its fields, unchanged, to `data/maude/cleaned/insulin_pumps_{year}.jsonl`. The output is ~5 KB per report and ~20 GB in total, too big to load a year at once, so each file has one report per line and the steps below read it line by line. Takes ~30 min.

In [ ]:
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
for f in CLEAN_DIR.glob('insulin_pumps_*'):
    f.unlink()


def is_pump(r: dict) -> bool:
    return any(PUMP.search(d.get('generic_name') or '') for d in r.get('device') or [])


counts = {}
for year, group in parts.groupby(parts.period.str.split('q').str[0]):
    out = CLEAN_DIR / f'insulin_pumps_{year}.jsonl'
    n = 0
    with out.open('w', encoding='utf-8') as fh:
        for path in group.path:
            for r in filter(is_pump, load_partition(path)):
                fh.write(json.dumps(r) + '\n')
                n += 1
    if n:
        counts[year] = n
    else:
        out.unlink()
    print(f'{year}: {n:,} pump reports')

pd.Series(counts, name='reports')

## 3. ID

Give every report an `id`: a sequential integer, unique across all files. Rewrites the files in place; re-running assigns the same ids.

In [ ]:
def read_reports(f: Path):
    with f.open(encoding='utf-8') as fh:
        yield from map(json.loads, fh)


files = sorted(CLEAN_DIR.glob('insulin_pumps_*.jsonl'))
next_id = 0
for f in files:
    tmp = f.with_suffix('.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        for r in read_reports(f):
            r.pop('id', None)
            fh.write(json.dumps({'id': next_id, **r}) + '\n')
            next_id += 1
    tmp.replace(f)
    print(f'{f.name}: ids up to {next_id - 1:,}')

## 4. Columns

Every field name across all reports. Nested fields are dotted (`device.brand_name`, `patient.patient_problems`). `present` counts the reports that have the field at all and `non_empty` the ones where its value isn't blank.

In [ ]:
def has_value(v: object) -> bool:
    if isinstance(v, str):
        return v.strip() != ''
    if isinstance(v, list):
        return any(map(has_value, v))
    return v is not None


def fields(v: object, prefix: str = ''):
    if isinstance(v, dict):
        for k, x in v.items():
            yield from fields(x, f'{prefix}.{k}' if prefix else k)
    elif isinstance(v, list) and any(isinstance(x, dict) for x in v):
        for x in v:
            yield from fields(x, prefix)
    else:
        yield prefix, has_value(v)


present, non_empty = Counter(), Counter()
total = 0
for f in files:
    for r in read_reports(f):
        seen = dict.fromkeys((k for k, _ in fields(r)), False)
        for k, v in fields(r):
            seen[k] |= v
        present.update(seen.keys())
        non_empty.update(k for k, v in seen.items() if v)
        total += 1

columns = pd.DataFrame({'present': present, 'non_empty': non_empty}).fillna(0).astype(int)
columns['% non_empty'] = (columns.non_empty / total * 100).round(1)
print(f'{len(columns)} columns over {total:,} reports')
with pd.option_context('display.max_rows', None):
    display(columns.sort_index())

## 5. Trim

Keep only the fields about the report, the device, the patient and outcome, and what went wrong; drop manufacturer and distributor contact details, database bookkeeping and always-empty fields. Rewrites the files in place; undo by re-running steps 2–3.

In [ ]:
TOP = [
    'id',
    'adverse_event_flag',
    'product_problem_flag',
    'product_problems',
]
NESTED = {
    'device': [
        'generic_name',
    ],
    'patient': [
        'patient_age',
        'patient_sex',
        'patient_problems',
    ],
    'mdr_text': ['text_type_code', 'text'],
}


def trim(r: dict) -> dict:
    out = {k: r[k] for k in TOP if k in r}
    for k, subs in NESTED.items():
        out[k] = [{s: x[s] for s in subs if s in x} for x in r.get(k) or []]
    return out


for f in files:
    before = f.stat().st_size
    tmp = f.with_suffix('.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        for r in read_reports(f):
            fh.write(json.dumps(trim(r)) + '\n')
    tmp.replace(f)
    print(f'{f.name}: {before / 1e6:,.0f} MB -> {f.stat().st_size / 1e6:,.0f} MB')

## 6. Flatten

One flat record per report: device and patient fields move to the top level, and the event narrative becomes `event_description`. A field a report lacks is `null`. Run once on trimmed files; undo by re-running steps 2–5.

In [ ]:
TEXT = {'Description of Event or Problem': 'event_description'}


def unnest(r: dict) -> dict:
    # <0.1% of reports list several devices or patients: keep the matched pump and the first patient.
    dev = next(d for d in r.pop('device') if PUMP.search(d.get('generic_name') or ''))
    pat = (r.pop('patient') or [{}])[0]
    text: dict[str, list[str]] = {}
    for t in r.pop('mdr_text'):
        if t['text_type_code'] in TEXT:
            text.setdefault(TEXT[t['text_type_code']], []).append(t['text'])
    return {
        **{k: r.get(k) for k in TOP},
        **{k: dev.get(k) for k in NESTED['device']},
        **{k: pat.get(k) for k in NESTED['patient']},
        **{c: '\n\n'.join(text[c]) if c in text else None for c in TEXT.values()},
    }


for f in files:
    tmp = f.with_suffix('.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        for r in read_reports(f):
            fh.write(json.dumps(unnest(r)) + '\n')
    tmp.replace(f)
    print(f'{f.name}: {f.stat().st_size / 1e6:,.0f} MB')

pd.read_json(files[-1], lines=True, nrows=5)

## 7. Filter

Drop, in place, every report where:

- any value (or list item) is `null` or a placeholder: NA, N/A, null, none, unknown, unk, not applicable, no information, insufficient information, invalid data, `*`, or an "Unknown (...)" code. Blank strings and empty lists are kept: most fields are optional.
- the patient had no consequences: "No Consequences Or Impact To Patient", "No Known Impact Or Consequence To Patient", "No Clinical Signs, Symptoms or Conditions"
- the product problem is "Adverse Event Without Identified Device or Use Problem" or "Insufficient Device Problem Information"

In [ ]:
PLACEHOLDER = re.compile(
    r'(n/?a|null|none|unk(nown)?|not applicable|no information|insufficient information'
    r'|invalid data|\*)|unknown \(.*\)',
    re.IGNORECASE,
)
NO_HARM = re.compile(r'^no (known )?(consequences|impact|clinical signs)', re.IGNORECASE)
NO_DEVICE_PROBLEM = {
    'Adverse Event Without Identified Device or Use Problem',
    'Insufficient Device Problem Information',
}


def keep(r: dict) -> bool:
    values = [x for v in r.values() for x in (v if isinstance(v, list) else [v])]
    return not (
        any(x is None or PLACEHOLDER.fullmatch(str(x).strip()) for x in values)
        or any(NO_HARM.match(p.strip()) for p in r['patient_problems'])
        or NO_DEVICE_PROBLEM & set(r['product_problems'])
    )


kept = {}
for f in files:
    tmp = f.with_suffix('.tmp')
    before = after = 0
    with tmp.open('w', encoding='utf-8') as fh:
        for r in read_reports(f):
            before += 1
            if keep(r):
                fh.write(json.dumps(r) + '\n')
                after += 1
    tmp.replace(f)
    kept[f.stem] = (before, after)

summary = pd.DataFrame(kept, index=['before', 'after']).T
print(f'kept {summary.after.sum():,} of {summary.before.sum():,} reports')
summary

## 8. Tidy text

The FDA export replaces characters it can't encode (mostly curly quotes and dashes) with `¿`, and leaves a few control characters and no-break spaces. Remove those from every text value; real symbols (®, °, ½, accents) stay.

In [ ]:
APOSTROPHE = re.compile(r'(?<=\w)¿(?=\w)')  # CUSTOMER¿S
JUNK = re.compile(r'[¿\x00-\x09\x0b-\x1f\x7f]')
SPACES = re.compile(r'[ \xa0]+')


def tidy(v: object) -> object:
    if isinstance(v, str):
        return SPACES.sub(' ', JUNK.sub(' ', APOSTROPHE.sub("'", v))).strip()
    if isinstance(v, list):
        return [tidy(x) for x in v]
    return v


for f in files:
    tmp = f.with_suffix('.tmp')
    with tmp.open('w', encoding='utf-8') as fh:
        for r in read_reports(f):
            fh.write(json.dumps({k: tidy(v) for k, v in r.items()}) + '\n')
    tmp.replace(f)

pd.read_json(files[-1], lines=True, nrows=5)